# Phase 3 — LLC Estimation

Computes LLC + drLLC (add) + drLLC (mult) for **one (ratio, seed)** pair.
Each run processes 100 checkpoints × 3 estimates × 8 chains ≈ **3–7 h on T4**.

**Kaggle setup:**
1. Add **Phase 1** output as dataset input (has checkpoints)
2. Add **Phase 2** output as dataset input (has `llc_calibration.yaml`)
3. Set `RATIO` and `SEED` in Section 2
4. Enable GPU T4 + Internet → *Save and Run All*

Run this notebook once per (ratio, seed) pair. The LLC CSV is resumable —
already-computed epochs are skipped automatically.

## Section 0 — Setup

In [ ]:
import os, sys, shutil, subprocess

PLATFORM = "kaggle"   # "kaggle" or "colab"
REPO_URL  = "https://github.com/makataomu/slt-diplomka"

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
    REPO_DIR    = "/content/slt"
    PERSIST_DIR = "/content/drive/MyDrive/slt_persist"
    for d in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{PERSIST_DIR}/{d}", exist_ok=True)
else:
    REPO_DIR    = "/kaggle/working/slt"
    PERSIST_DIR = None

if os.path.exists(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Pulled latest from GitHub")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Cloned from GitHub")

if PLATFORM == "colab":
    lnk = f"{REPO_DIR}/results"
    if os.path.islink(lnk): os.unlink(lnk)
    elif os.path.isdir(lnk): shutil.rmtree(lnk)
    os.symlink(f"{PERSIST_DIR}/results", lnk)
    print(f"results/ -> {PERSIST_DIR}/results")
else:
    for sub in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{REPO_DIR}/{sub}", exist_ok=True)
    # Restore checkpoints + existing LLC CSVs from all dataset inputs
    for inp in sorted(os.listdir("/kaggle/input")):
        prev = f"/kaggle/input/{inp}/slt/results"
        if os.path.exists(prev):
            print(f"Restoring results from /kaggle/input/{inp}/ ...")
            for sub in ["checkpoints", "metrics"]:
                src, dst = f"{prev}/{sub}", f"{REPO_DIR}/results/{sub}"
                if os.path.exists(src):
                    for item in os.listdir(src):
                        s, d = f"{src}/{item}", f"{dst}/{item}"
                        if not os.path.exists(d):
                            (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
            print("  Done.")

# Restore calibration yaml (looks in all dataset inputs)
calib_local = f"{REPO_DIR}/configs/llc_calibration.yaml"
if PLATFORM == "colab":
    src = f"{PERSIST_DIR}/llc_calibration.yaml"
    if os.path.exists(src): shutil.copy(src, calib_local)
else:
    for inp in sorted(os.listdir("/kaggle/input")):
        src = f"/kaggle/input/{inp}/llc_calibration.yaml"
        if os.path.exists(src):
            shutil.copy(src, calib_local)
            print(f"Restored llc_calibration.yaml from /kaggle/input/{inp}/")
            break

os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
print(f"\nReady. Platform={PLATFORM} | cwd={os.getcwd()}")

# Sanity check calibration
import yaml
cfg = yaml.safe_load(open("configs/llc_calibration.yaml"))
if cfg.get("calibrated"):
    print(f"Calibration OK: epsilon={cfg['epsilon']}  nbeta={cfg['nbeta']}  gamma={cfg['gamma']}")
else:
    print("WARNING: llc_calibration.yaml not calibrated — run Phase 2 first.")


## Section 1 — Install dependencies

In [ ]:
%pip install -q transformer_lens devinterp zarr==3.1.6

import importlib.metadata, torch
print(f"devinterp {importlib.metadata.version('devinterp')}  "
      f"| torch {torch.__version__}  "
      f"| device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


## Section 2 — Configure and run

Set `RATIO` and `SEED`, then run. Already-completed epochs are skipped automatically.

In [ ]:
RATIO = 0.50   # one of [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
SEED  = 0      # 0, 1, or 2

!python src/llc_estimation.py --ratio {RATIO} --seed {SEED}


## Section 3 — Diagnostics

In [ ]:
from pathlib import Path
import pandas as pd

print("=== LLC metrics ===")
for f in sorted(Path("results/metrics").glob("*_llc.csv")):
    df = pd.read_csv(f)
    print(f"  {f.name}: {len(df)} epochs done")
    if len(df):
        print(f"    LLC range: [{df['LLC'].min():.3f}, {df['LLC'].max():.3f}]")
